last update: JohsienY. March 31, 2025

## Package import and some helpful function

In [ ]:
# import python packages, important to keep in this enviornment

import gspread
from google.auth import default

import h5py
import math
import numpy as np
import os
import pandas as pd
import glob

from scipy.interpolate import interp1d
from scipy.spatial import ConvexHull
from sklearn.metrics import r2_score
from skimage.transform import resize

In [ ]:
def read_metadata(gsheet_url,tab_name):
  """This script reads information in a tab in the Google sheet, where metadata for each experiment is stored.
  """
  # import gspread
  # from google.auth import default

  # reading in google sheet with experimental details
  creds, _ = default()
  gc = gspread.authorize(creds)

  # insert URl of gsheet, specify worksheets as different variables
  wb = gc.open_by_url(gsheet_url)
  exps = wb.worksheet(tab_name)

  #
  all_data = pd.DataFrame(exps.get_all_records())
  metadata = all_data.set_index('experiment')
  display(all_data)

  return metadata

# math stuff

In [ ]:
# basic processing
def fill_missing(Y, kind="linear"):
    """Fills missing values independently along each dimension after the first.
    Args:
      Y: array like object
      kind: 'linear', refer to scipy.interpolate.interp1d for more options
    Returns:
      np.array
    """
    # Store initial shape.
    initial_shape = Y.shape
    # Flatten after first dim.
    Y = Y.reshape((initial_shape[0], -1))
    # Interpolate along each slice.
    for i in range(Y.shape[-1]):
        y = Y[:, i]
        # Build interpolant.
        x = np.flatnonzero(~np.isnan(y))
        f = interp1d(x, y[x], kind=kind, fill_value=np.nan, bounds_error=False)
        # Fill missing
        xq = np.flatnonzero(np.isnan(y))
        y[xq] = f(xq)
        # Fill leading or trailing NaNs with the nearest non-NaN values
        mask = np.isnan(y)
        y[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), y[~mask])
        # Save slice
        Y[:, i] = y
    # Restore to initial shape.
    Y = Y.reshape(initial_shape)
    return Y

In [ ]:
def running_mean(x, N):
    return np.convolve(x, np.ones(N)/N, mode='same')

def normalize_array(inputs):
    output = (inputs-min(inputs))/(max(inputs)-min(inputs))
    return output

In [ ]:
def find_nearest(ref_array, array2align):
  """Find the indices of element in a template array
   whose values are the closest to the ones in a array of interest
  Args:
    ref_array: array as template
    array2align: array whose elements need to be aligned to a reference
  Returns:
    array_idx: array of indices (length of array2align)
  """
  align_idx = []
  for instances in array2align:
    idx = (np.absolute(ref_array - instances)).argmin()
    align_idx.append(idx)
  array_idx = np.array(align_idx)
  return array_idx

In [ ]:
def speed_angular_sp(x,y,heading,body_length,frame_rate,scale):
  """ Swimming kinematics of given xy positions over time
  Args:
    x, y (float): arrays of xy coordinates in pixel
    heading (float): arrays of heading direction in degree
    scale (float): value of mm per pixel in experiment video
    body_length (float): arrays of each fish body
    frame_rate (float): video frame rate in Hz
  Returns:
    speed: array of speed and angular speed in px/sec
    angular speed: array of angular speed in dg/sec
  """
  max_speed = 20*body_length # per second
  max_ang_speed = 720 # per second

  # use position difference across 2 inter-frame intervals
  dx = x.flat[2:] - x.flat[:-2]
  dx = np.pad(dx, (1, 1), 'constant', constant_values=(0, 0)) # pad 0 to both ends
  dy = y.flat[2:] - y.flat[:-2]
  dy = np.pad(dy, (1, 1), 'constant', constant_values=(0, 0)) # pad 0 to both ends
  sp = np.sqrt(dx**2+dy**2)/scale*(frame_rate/2)
  sp_filt = np.where(sp > max_speed, np.nan, sp)
  speed = fill_missing(sp_filt)

  # use orientation difference between two consecutive frames
  dag = np.ediff1d(heading, to_begin=0)
  asp=np.array([((d+180)%360 - 180)*frame_rate for d in dag]) #mlb change
  #asp = ((dag+180)%360 - 180)*frame_rate
  asp_filt = np.where(abs(asp) > max_ang_speed, np.nan, asp)
  ang_speed = fill_missing(asp_filt)
  return speed, ang_speed

# calculate angles between 2 points
def calculate_angle(X1,Y1,X2,Y2):
    rads=[];
    for x1,y1,x2,y2 in zip(X1,Y1,X2,Y2):
        dx=x1-x2
        dy=y1-y2
        rads.append(math.atan2(dy, dx))
    degs = np.rad2deg(rads)
    return degs

def get_single_angle(x1,y1,x2,y2):
    return math.degrees(math.atan2(y2-y1, x2-x1))

In [ ]:
def pt_rotate(x, y, deg, self_x, self_y):
    """Rotate a point around a given point.
    Args:
      x, y: arrays of xy coordinates that need to rotate
      self_x, self_y: arrays of xy coordinates that serve as reference point
      deg: orientation angle of the reference point
    Returns:
      qx, qy: new xy coordinates when reference point is at (0,0) facing north
    """
    qx = self_x + math.cos(np.deg2rad(deg))*(x-self_x) + math.sin(np.deg2rad(deg))*(y-self_y)
    qy = self_y - math.sin(np.deg2rad(deg))*(x-self_x) + math.cos(np.deg2rad(deg))*(y-self_y)
    return qx, qy

In [ ]:
def fish_polygon_area(xs,ys):
    """Calculate the area of a polygon given its vertices."""
    xs=np.array(xs).T
    ys=np.array(ys).T
    areas=[]
    for f_x,f_y in zip(xs,ys):
      points=[[x,y] for x,y in zip(f_x,f_y)]
      hull=ConvexHull(np.array(points))
      areas.append(hull.area)

    return areas

def fish_polygon_area2(xs,ys):
    """Calculate the area of a polygon given its vertices."""
    xs=np.vstack(xs).T
    ys=np.vstack(ys).T
    areas=[]
    for f_x,f_y in zip(xs,ys):
      points = np.column_stack((f_x,f_y))
      hull = ConvexHull(np.array(points))
      areas.append(hull.area)

    return areas

# basic behavior

In [ ]:
class Fish():
  """Collect individual fish tracking info for easier  data call in pipeline.
  """
  pass

def process_h5(h5tracks,fish_num,sleap_nodes,exp):
  """This script reads xy info of each body node of individual fish in a group.
  Return: list of objects with attributes of xy coordinates.
  """
  frame_num, _, _, track_num = h5tracks.shape
  group_data = []
  for f in range(fish_num):

    name = exp + '_f'+ str(f+1)
    fish = Fish() # make class object
    setattr(fish,'name',name)

    # load all xy coordinates
    for idx, node in enumerate(sleap_nodes):
      setattr(fish,str(node)+'x',fill_missing(h5tracks[:,idx,0,f]))
      setattr(fish,str(node)+'y',fill_missing(h5tracks[:,idx,1,f]))
    group_data.append(fish)

  return group_data

In [ ]:
def group_arrays(group_data,fish_num,frame_num,frame_rate,scale):
  """This script takes the processed h5 data and stacks them into group arrays shaped in fish_num x frame_num.
  """
  # initialize group arrays
  f_bodylength_mm = [[] for f in range(fish_num)]
  f_bodylength_px = [[] for f in range(fish_num)]

  # postional (mlb updated)
  f_nosex = [[] for f in range(fish_num)]; f_nosey = [[] for f in range(fish_num)] # center of two eyes
  f_x = [[] for f in range(fish_num)]; f_y = [[] for f in range(fish_num)] # front bladder or belly as the centroid
  f_tailx = [[] for f in range(fish_num)]; f_taily = [[] for f in range(fish_num)] # tip of the tail
  f_rbx = [[] for f in range(fish_num)]; f_rby = [[] for f in range(fish_num)] # rear bladder
  f_t1x = [[] for f in range(fish_num)]; f_t1y = [[] for f in range(fish_num)] # tail point 1
  f_t2x = [[] for f in range(fish_num)]; f_t2y = [[] for f in range(fish_num)] # tail point 2
  f_t12x = [[] for f in range(fish_num)]; f_t12y = [[] for f in range(fish_num)] # center of tail point 1 and 2
  # angular
  f_heading = [[] for f in range(fish_num)]; f_tail_angle = [[] for f in range(fish_num)]
  f_tail_angle0 = [[] for f in range(fish_num)]; f_tail_angle1 = [[] for f in range(fish_num)]
  f_tail_angle2 = [[] for f in range(fish_num)]; f_tail_angle3 = [[] for f in range(fish_num)]
  # speed stuff
  f_speed = [[] for f in range(fish_num)]; f_ang_speed = [[] for f in range(fish_num)]

  # set window size (num of frame) for running mean
  pos_ker = 5 # for positional data
  ang_ker = 5 # for angular data
  spd_ker = 10 # for speed data

  # loop through the individual fish
  for ndx, fish in enumerate(group_data):

    # get ind nose - centr of two eyes
    nosex = [int(np.mean([l,r])) for l,r in zip(fish.lex,fish.rex)]
    nosey = [int(np.mean([l,r])) for l,r in zip(fish.ley,fish.rey)]

    # get ind fish body length - 95th percentil across the frames
    nose_to_tail = np.sqrt( (nosex - fish.ttx)**2 + (nosey - fish.tty)**2 )
    bodylen_px =  np.percentile(nose_to_tail,99)
    bodylen_mm =  np.percentile(nose_to_tail,99)*scale

    # stack the convolved arrays
    f_bodylength_px[ndx] = bodylen_px
    f_bodylength_mm[ndx] = bodylen_mm

    f_nosex[ndx] = running_mean(nosex,pos_ker)
    f_nosey[ndx] = running_mean(nosey,pos_ker)
    f_x[ndx] = running_mean(fish.mbx,pos_ker) #if jy nodes
    f_y[ndx] = running_mean(fish.mby,pos_ker) #if jy nodes
    f_tailx[ndx] = running_mean(fish.ttx,pos_ker)
    f_taily[ndx] = running_mean(fish.tty,pos_ker)

    # test other nodes
    f_rbx[ndx] = running_mean(fish.rbx,pos_ker); f_rby[ndx] = running_mean(fish.rby,pos_ker)
    f_t1x[ndx] = running_mean(fish.t1x,pos_ker); f_t1y[ndx] = running_mean(fish.t1y,pos_ker)
    f_t2x[ndx] = running_mean(fish.t2x,pos_ker); f_t2y[ndx] = running_mean(fish.t2y,pos_ker)
    t12x = [int(np.mean([l,r])) for l,r in zip(fish.t1x,fish.t2x)]
    t12y = [int(np.mean([l,r])) for l,r in zip(fish.t1y,fish.t2y)]
    f_t12x[ndx] = running_mean(t12x,pos_ker); f_t12y[ndx] = running_mean(t12y,pos_ker)

    # get ind fish orientation and tail angle per frame
    heading = calculate_angle(f_nosex[ndx],f_nosey[ndx],f_x[ndx],f_y[ndx])
    tail = calculate_angle(f_x[ndx],f_y[ndx],f_tailx[ndx],f_taily[ndx])
    f_heading[ndx] = running_mean(heading, ang_ker)
    f_tail_angle[ndx] = running_mean((( (tail-heading) + 180) % 360) - 180, ang_ker)

    # get ind fish speed and angular speed per frame
    speed, angular_speed = speed_angular_sp(fish.mbx,fish.mby,heading,bodylen_px,frame_rate,scale)
    f_speed[ndx] = running_mean(speed, spd_ker)
    f_ang_speed[ndx] = running_mean(angular_speed, spd_ker)

  # MAKE ALL ARRAYS FISH_NUM X TIMEPOINTS
  f_nosex = np.array(f_nosex); f_nosex = np.vstack(f_nosex)
  f_nosey = np.array(f_nosey); f_nosey = np.vstack(f_nosey)
  f_x = np.array(f_x); f_x = np.vstack(f_x)
  f_y = np.array(f_y); f_y = np.vstack(f_y)
  f_tailx = np.array(f_tailx); f_tailx = np.vstack(f_tailx)
  f_taily = np.array(f_taily); f_taily = np.vstack(f_taily)
  f_heading = np.array(f_heading); f_heading = np.vstack(f_heading)
  f_tail_angle = np.array(f_tail_angle); f_tail_angle = np.vstack(f_tail_angle)
  f_speed = np.array(f_speed)
  f_ang_speed = np.array(f_ang_speed)


  return f_bodylength_px, f_bodylength_mm, f_nosex, f_nosey, f_x, f_y, f_tailx, f_taily, f_heading, f_tail_angle, f_speed, f_ang_speed

In [ ]:
def analyze_neighbors(fish_num,f_heading,f_x,f_y,frame_num,scale,f_body_length):
  """This script generates arrays of group level dynamic (distance, alignment, etc.).
  """

  if fish_num == 2:

    # initialize arrays and lists
    ff_dist = np.zeros((2,1,frame_num))
    ff_align = np.zeros((2,1,frame_num))
    f_IID = np.zeros((1,frame_num));
    f_IIA = np.zeros((1,frame_num));
    f_closest_id = np.zeros((1,frame_num))
    f_closest_dist = np.zeros((1,frame_num))
    f_closest_align = np.stack( (np.ones((frame_num)), np.zeros((frame_num))) )

    # there's only one pair
    dist = np.abs(np.sqrt( (f_x[0]-f_x[1])**2 + (f_y[0]-f_y[1])**2 ) )*scale
    align = ((( f_heading[0]-f_heading[1] ) + 180) % 360 - 180)

    for f0 in range(fish_num):
      ff_dist[f0] = dist; ff_align[f0] = align
    dist = dist.reshape(1,frame_num); align = align.reshape(1,frame_num)
    f_IID = dist; f_closest_dist = dist
    f_IIA = align; f_closest_align = align

  else: # if fish_num >= 3

    # initialize arrays and lists
    ff_dist = np.zeros((fish_num,fish_num-1,frame_num))
    ff_align = np.zeros((fish_num,fish_num-1,frame_num))
    f_IID = []; f_IIA = [] # number of dyads x framenum
    f_closest_id = np.zeros((fish_num,frame_num))
    f_closest_dist = np.zeros((fish_num,frame_num))
    f_closest_align = np.zeros((fish_num,frame_num))

    # loop through ind fish
    for f0 in range(fish_num):
      nb_count = 0

      for f1 in range(fish_num):
        if f1 != f0:
          nb_count += 1

          dist = np.abs(np.sqrt( (f_x[f0]-f_x[f1])**2 + (f_y[f0]-f_y[f1])**2 ) )*scale
          align = ((( f_heading[f0]-f_heading[f1] ) + 180) % 360 - 180)

          ff_dist[f0][nb_count-1] = dist
          ff_align[f0][nb_count-1] = align


        if f0 < f1:
          f_IID.append(dist)
          f_IIA.append(align)

    f_IID = np.array(f_IID)
    f_IIA = np.array(f_IIA)

    # find the stats with the closest neighbor for each fish
    for f0 in range(fish_num):
      closest_dist = np.array(np.min(ff_dist[f0],axis=0)) # find the smallest dist in each frame
      f_closest_id[f0] = np.argmin(ff_dist[f0] == closest_dist, axis=0) # identify which fish it is
      index = f_closest_id[f0].astype(int)
      f_closest_dist[f0] = closest_dist # find their distance
      f_closest_align[f0] = ff_align[f0][index,np.arange(len(index))] # find their alignment

  return ff_dist, ff_align, f_IID, f_IIA, f_closest_id, f_closest_dist, f_closest_align

In [ ]:
def moving_frames(fish_num,f_speed,f_bodylength,threshold=2):
  """This script generates boolean arrays of whether individual fish is moving
  """
  if fish_num == 2:

    # make binary array to filter whether each fish is moving in every frame
    # f_moving = np.zeros((fish_num,f_speed.shape[-1]))
    moving = []

    for i,j in zip(f_speed,f_bodylength):
      mov = []
      mov.append([1 if d/j > threshold else 0 for d in i])
      moving.append(np.array(mov))

    f_moving = np.array(moving) # fish_num x frame_num array
    f_moving = np.squeeze(f_moving)

    # filter when both fish in each pair are both moving
    pairs_moving = f_moving[0]*f_moving[1]
    pairs_moving = np.squeeze(pairs_moving)

  else: # more than 2 fish

    # make binary array to filter whether each fish is moving in every frame
    moving = []

    for i,j in zip(f_speed,f_bodylength):
      mov = []
      mov.append([1 if d/j > threshold else 0 for d in i])
      moving.append(np.array(mov))

    f_moving = np.array(moving) # fish_num x frame_num array
    f_moving = np.squeeze(f_moving)

    # filter when both fish in each pair are both moving
    # all_pair_moving = np.zeros((fish_num,fish_num-1,f_speed.shape[-1]))
    pairs_moving = []

    for f0 in range(fish_num):
      nb_count = 0
      for f1 in range(fish_num):
        if f1 != f0:
          nb_count += 1
          m = f_moving[f0]*f_moving[f1]
          # all_pair_moving[f0][nb_count-1] = m
          if f0 < f1:
            pairs_moving.append(m)

    pairs_moving = np.array(pairs_moving) # pair_num x frame_num array
    pairs_moving = np.squeeze(pairs_moving)

    # group_moving = np.all(moving, axis=0) # 1 x frame_num array
    # all_moving = np.tile(group_moving,(fish_num,1)) # fish_num x frame_num

  return f_moving, pairs_moving

# more processing

In [ ]:
def get_visual_fields(fish_num,f_x,f_y,f_heading,f_bodylength,start,stop,scale,sample):
  """This scripts generates visual fields for each fish.
  Returns:
    occupied_angles: 3D arrays (fish_num,fish_num-1,frames)
    subtended_angles: 3D arrays (fish_num,fish_num-1,frames)
  """
  sample = sample # sampling rate - take every Nth frames
  occupied_angles=[]

  # for each focal fish
  for f0 in range(fish_num):
    fish_tmp = []
    # loop through each neighbor
    for f1 in range(fish_num):
      if f1 != f0:
        tmp=[]

        #fish details
        x_0, y_0 = f_x[f0][start:stop:sample], f_y[f0][start:stop:sample]
        heading_0 = f_heading[f0][start:stop:sample]
        x_1, y_1 = f_x[f1][start:stop:sample], f_y[f1][start:stop:sample]
        heading_1 = f_heading[f1][start:stop:sample]
        bodylength_1 = f_bodylength/scale

        for ndx,(d0,d1,x0,x1,y0,y1) in enumerate(zip(heading_0,heading_1,x_0,x_1,y_0,y_1)):
            x1_rot, y1_rot = pt_rotate(x1, y1, d0, x0, y0)

            # Calculate the position of the endpoints of the line
            endpoint1_x = x1_rot + bodylength_1/2.0 * np.cos(d1)
            endpoint1_y = y1_rot + bodylength_1/2.0 * np.sin(d1)

            endpoint2_x = x1_rot - bodylength_1/2.0 * np.cos(d1)
            endpoint2_y = y1_rot - bodylength_1/2.0 * np.sin(d1)

            angle1 = np.arctan2(endpoint1_y, endpoint1_x)
            angle2 = np.arctan2(endpoint2_y, endpoint2_x)

            tmp.append([min(angle1, angle2), max(angle1, angle2)])

        fish_tmp.append(tmp)
        del(tmp)
    occupied_angles.append(fish_tmp)
    del(fish_tmp)
  occupied_angles = np.array(occupied_angles)

  # get the subtended angle for each fish
  subtend_angs=[]
  for f0 in occupied_angles: # loop through focal fish
    single_view=np.zeros(len(f0[0]))
    sub_view=[]
    for f1 in f0: # loop through neighbor fish
      # get the subtended angle for each fish
      sub = [abs(m2 - m1) if abs(m2-m1) < 180 else 360-(m2-m1) for m1, m2 in zip(f1[:,0],f1[:,1])]
      sub_view.append(sub)

      for ndx,a in enumerate(f1):
        # get L-R angle diff
        if abs(a[1]-a[0])>0: # make sure it's not zero
          tmpl=0;tmpr=0
          if a[1]>0 and a[0]>0: tmpl=a[1]-a[0]
            # entire fish in left visual field
          if a[1]<0 and a[0]<0: tmpr=a[1]-a[0]
            # entire fish in right visual field
          if a[1]*a[0]<0 and abs(a[1]-a[0])<180: tmpl=abs(a[1]); tmpr=abs(a[0])
            # neighbor fish cross midline in the front
          if a[1]*a[0]<0 and abs(a[1]-a[0])>180: tmpl=abs(180-a[0]); tmpr=abs(-180-a[1])
            # neighbor fish cross midline in the back
          if abs(a[1]-a[0])==0: tmpl=0;tmpr=0
            # unlikely but if the neighbor fish is perfectly align in midline
          single_view[ndx]+=tmpl-tmpr
    subtend_angs.append(sub_view)
  subtend_angs = np.array(subtend_angs)

  return occupied_angles, subtend_angs